华为云：https://pangu.huaweicloud.com/gallery/asset-detail.html?id=9e09753d-fb42-4f2a-bca5-28ab2c07697b

# 基于Mask2Former模型进行推理
Mask2Former 是 Meta AI 的一个非常好的新模型，能够使用相同的架构解决任何类型的图像分割（无论是实例、语义还是全景分割）。该模型在 DETR 和 MaskFormer 的基础上进行了改进，在 Transformer 解码器中加入了掩蔽注意力。


我们将使用该模型进行全景、语义和实例分割的推理（即对新图像进行预测）。

## 环境搭建

首先，我们需要安装`python`, `mindnlp` 和 `mindspore`。

In [ ]:
%%capture captured_output
!/home/ma-user/anaconda3/bin/conda create -n python-3.9.0 python=3.9.0 -y --override-channels --channel https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
!/home/ma-user/anaconda3/envs/python-3.9.0/bin/pip install ipykernel

In [ ]:
import json
import os

data = {
   "display_name": "python-3.9.0",
   "env": {
      "PATH": "/home/ma-user/anaconda3/envs/python-3.9.0/bin:/home/ma-user/anaconda3/envs/python-3.7.10/bin:/modelarts/authoring/notebook-conda/bin:/opt/conda/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/home/ma-user/modelarts/ma-cli/bin:/home/ma-user/modelarts/ma-cli/bin"
   },
   "language": "python",
   "argv": [
      "/home/ma-user/anaconda3/envs/python-3.9.0/bin/python",
      "-m",
      "ipykernel",
      "-f",
      "{connection_file}"
   ]
}

if not os.path.exists("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/"):
    os.mkdir("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/")

with open('/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/kernel.json', 'w') as f:
    json.dump(data, f, indent=4)

In [ ]:
#%%capture captured_output

!pip install https://ms-release.obs.cn-north-4.myhuaweicloud.com/2.2.14/MindSpore/unified/x86_64/mindspore-2.2.14-cp39-cp39-linux_x86_64.whl --trusted-host ms-release.obs.cn-north-4.myhuaweicloud.com -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install mindnlp

## 定义模型

我们将实例化 Mask2Former 模型及其图像处理器。

我们加载了一个在 COCO 全景数据集上训练的 Mask2Former 模型。


In [ ]:
from mindnlp.transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

processor = AutoImageProcessor.from_pretrained("facebook/mask2former-swin-base-coco-panoptic")
model = Mask2FormerForUniversalSegmentation.from_pretrained("facebook/mask2former-swin-base-coco-panoptic")

## 导入图像

使用一张猫的图像，它来源于 COCO 数据集。

In [ ]:
from PIL import Image
import requests

url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)
image

我们使用图像处理器对图像进行处理。

In [ ]:
inputs = processor(images=image, return_tensors="ms")
for k,v in inputs.items():
    print(k,v.shape)

## 前向传播


接下来，我们可以通过模型计算**像素**和**像素掩码**。我们使用`mindsport.set_text（device_target='GPU')`，不需要计算梯度（这仅在训练模型时有用）。

In [ ]:
import mindspore;mindspore.set_context(device_target='GPU');mindspore.run_check()

In [ ]:
outputs = model(**inputs)

## 可视化

对于可视化，我们可以对模型的输出进行后处理。在全景分割的情况下，该模型对每张图像预测：1.分割图；2.相应的segments_info.

In [ ]:
# you can pass them to processor for postprocessing
results = processor.post_process_panoptic_segmentation(outputs, target_sizes=[image.size[::-1]])[0]
print(results.keys())

In [ ]:
for segment in results['segments_info']:
    print(segment)

对于这个例子下，模型能够识别出图像中的5个不同对象。

In [ ]:
segment_to_label = {segment['id']: segment['label_id'] for segment in results["segments_info"]}
print(segment_to_label)

让我们可视化第一部分的二值化掩码:

In [ ]:
import numpy as np

def get_mask(segment_id):
  print("Visualizing mask for:", model.config.id2label[segment_to_label[segment_id]])

  mask = (results['segmentation'].numpy() == segment_id)
  visual_mask = (mask * 255).astype(np.uint8)
  visual_mask = Image.fromarray(visual_mask)

  return visual_mask

# note: segment with id == 0 means "background",
# so we visualize segment with id == 1 here
get_mask(segment_id=1)

为每个图像掩码创建单独的可视化图像：

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import cm
import mindspore.ops as ops
def draw_panoptic_segmentation(segmentation, segments_info):
    # get the used color map
    #ops.max(mindspore.tensor(results['segmentation'].numpy()))[0]
    viridis = cm.get_cmap('viridis', ops.max(mindspore.tensor(results['segmentation'].numpy()))[0])
    fig, ax = plt.subplots()
    ax.imshow(segmentation.numpy())
    instances_counter = defaultdict(int)
    handles = []
    # for each segment, draw its legend
    for segment in segments_info:
        segment_id = segment['id']
        segment_label_id = segment['label_id']
        segment_label = model.config.id2label[segment_label_id]
        label = f"{segment_label}-{instances_counter[segment_label_id]}"
        instances_counter[segment_label_id] += 1
        color = viridis(segment_id)
        handles.append(mpatches.Patch(color=color, label=label))
        
    ax.legend(handles=handles)

draw_panoptic_segmentation(**results)

我们可以看到该模型能够检测图像中的猫和遥控器。语义分割模型只会为“猫”类别创建一个单一的掩码。

## 推理（语义分割）

我们可以执行与上述相同的操作。在语义分割数据集 CityScapes 数据样本上进行推理。

In [ ]:
from mindnlp.transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

processor = AutoImageProcessor.from_pretrained("facebook/mask2former-swin-large-cityscapes-semantic")
model = Mask2FormerForUniversalSegmentation.from_pretrained("facebook/mask2former-swin-large-cityscapes-semantic")

导入一张 CityScapes 样本图像：

In [ ]:
from PIL import Image
import requests
# https://aistudio.baidu.com/datasetdetail/299692
url = 'https://ai-studio-online.bj.bcebos.com/v1/files/07b7cab878264e89bc012937194edebcaa3270af11bc45ecad690cbb85142605?responseContentDisposition=attachment%3Bfilename%3Droad.png&authorization=bce-auth-v1%2F5cfe9a5e1454405eb2a975c43eace6ec%2F2024-11-11T11%3A00%3A01Z%2F21600%2F%2Fef073f0dcdc13c29e724b78f339ec254b547df54e270b9a89834261ebc16a7c4'
image = Image.open(requests.get(url, stream=True).raw)
image

In [ ]:
inputs = processor(images=image, return_tensors="ms")
for k,v in inputs.items():
    print(k,v.shape)

In [ ]:
import mindspore;mindspore.set_context(device_target='GPU');mindspore.run_check()

In [ ]:
outputs = model(**inputs)

In [ ]:
# you can pass them to processor for postprocessing
predicted_map = processor.post_process_semantic_segmentation(outputs, target_sizes=[image.size[::-1]])[0]
print(predicted_map.shape)

我们可以在图像上绘制语义分割图（每个像素对应一个标签）：

In [ ]:
import numpy as np
# generate random color palette, which maps each class to a RGB value
color_palette = [list(np.random.choice(range(256), size=3)) for _ in range(len(model.config.id2label))]
print(color_palette)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

seg = predicted_map
color_seg = np.zeros((seg.shape[0], seg.shape[1], 3), dtype=np.uint8) # height, width, 3
palette = np.array(color_palette)
for label, color in enumerate(palette):
    # print(label)
    # print(color.shape)
    color_seg[(seg == label).numpy(), :] = color
# Convert to BGR
color_seg = color_seg[..., ::-1]

# Show image + mask
img = np.array(image) * 0.5 + color_seg * 0.5
img = img.astype(np.uint8)

plt.figure(figsize=(15, 10))
plt.imshow(img)
plt.show()

## 推理（实例分割）

In [ ]:
from mindnlp.transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

processor = AutoImageProcessor.from_pretrained("facebook/mask2former-swin-large-coco-instance")
model = Mask2FormerForUniversalSegmentation.from_pretrained("facebook/mask2former-swin-large-coco-instance")

In [ ]:
from PIL import Image
import requests

url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)
image

In [ ]:
inputs = processor(images=image, return_tensors="ms")
for k,v in inputs.items():
    print(k,v.shape)

In [ ]:
import mindspore;mindspore.set_context(device_target='GPU');mindspore.run_check()

In [ ]:
outputs = model(**inputs)

In [ ]:
outputs.keys()

In [ ]:
# you can pass them to processor for postprocessing
results = processor.post_process_instance_segmentation(outputs, target_sizes=[image.size[::-1]], threshold=0.9)[0]
print(results.keys())

In [ ]:
np.where(outputs.class_queries_logits.argmax(-1) == 65)

In [ ]:
outputs.masks_queries_logits[:,160,:,:]

In [ ]:
# you can pass them to processor for postprocessing
results = processor.post_process_instance_segmentation(outputs, target_sizes=[image.size[::-1]], threshold=0.9)[0]
print(results.keys())

In [ ]:
results['segmentation']

In [ ]:
for segment in results['segments_info']:
    print(segment)

In [ ]:
segment_to_label = {segment['id']: segment['label_id'] for segment in results["segments_info"]}
print(segment_to_label)

In [ ]:
import numpy as np

def get_mask(segment_id):
  print("Visualizing mask for:", model.config.id2label[segment_to_label[segment_id]])

  mask = (results['segmentation'].numpy() == segment_id)
  visual_mask = (mask * 255).astype(np.uint8)
  visual_mask = Image.fromarray(visual_mask)

  return visual_mask

# note: segment with id == 0 means "background",
# so we visualize segment with id == 1 here
get_mask(segment_id=1)

In [ ]:
get_mask(segment_id=2)

In [ ]:
get_mask(segment_id=3)

In [ ]:
get_mask(segment_id=4)